# 다음 포탈 크롤링 - 셀레니움 브라우저 제어


In [1]:
# 필요시 패키지 설치 (주석 해제 후 실행)
# !pip install selenium
# !pip install beautifulsoup4
# !pip install lxml
# !pip install pandas
# !pip install tqdm

In [2]:
import logging
import random
import glob
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
TimeoutException,
NoSuchElementException,
NoAlertPresentException,
StaleElementReferenceException,
WebDriverException
)
from bs4 import BeautifulSoup as bs
from IPython.display import display
import time
import pandas as pd
import re
import warnings
from tqdm import tqdm
from IPython.display import display
import os
SAVE_PATH = "./data_csv/"
OVERLAP_PATH = "./overlap_data"
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(OVERLAP_PATH, exist_ok=True)
warnings.filterwarnings(action="ignore")
# 로그 설정
logging.basicConfig(
level=logging.INFO,
format="%(asctime)s - %(levelname)s - %(message)s"
)
print("라이브러리 로드 완료")


라이브러리 로드 완료


## 1. 설정
### 1.1 키워드 및 수집 대상 설정
- 메인 키워드와 서브 키워드를 설정할 수 있어요
- 광고 관련 키워드를 포함한 게시물이 검색되지 않게 할 수 있어요
- 다음 포털 텍스트 수집 경로 5가지 중 원하는 요소를 수집할 수 있어요(기본값: 카페와 블로그)
- 백그라운드 실행 여부를 설정할 수 있어요 (True: 창 없이 백그라운드 실행 / False: 실제 브라우저 창 띄움)

In [3]:
# 백그라운드 실행 여부 설정 (True: 브라우저 창 없이 백그라운드 실행 / False: 실제 크롬 창 띄움)
USE_HEADLESS = True

# 메인 키워드 설정
main_keywords = ["귀농"]

# 투자 세부 키워드
invest_key = ["건강식", "아침"]

eliminate_ads_keyword = ""

# 다음 포털 텍스트 수집 가능 전체 경로 정의 (MECE 만족)
DAUM_TARGETS = {
    0: {"name": "다음카페", "w": "fusion", "col": "cafe"},
    1: {"name": "다음블로그_티스토리", "w": "fusion", "col": "tistory"},
    2: {"name": "다음블로그_브런치", "w": "fusion", "col": "brunch"},
    3: {"name": "다음웹문서", "w": "fusion", "col": "web"},
    4: {"name": "다음뉴스", "w": "news", "col": ""},
}

# 수집할 대상 번호 선택 (원하는 번호들을 리스트로 입력: 0, 1, 2, 3, 4)
SELECTED_TARGETS = [0, 1, 2] # 카페와 블로그를 수집하는게 기본값입니다.

print("========== 선택된 수집 대상 ==========")
for target_idx in SELECTED_TARGETS:
    if target_idx in DAUM_TARGETS:
        target_info = DAUM_TARGETS[target_idx]
        print(f"번호 [{target_idx}] -> {target_info['name']} 수집 대상으로 지정되었습니다.")
    else:
        print(f"번호 [{target_idx}] -> 유효하지 않은 번호입니다. (선택 가능 번호: {list(DAUM_TARGETS.keys())})")

========== 선택된 수집 대상 ==========
번호 [0] -> 다음카페 수집 대상으로 지정되었습니다.
번호 [1] -> 다음블로그_티스토리 수집 대상으로 지정되었습니다.
번호 [2] -> 다음블로그_브런치 수집 대상으로 지정되었습니다.


### 1.2 제외할 카페 설정

In [4]:
# 제외할 카페 목록 (필수 실행. 목록은 없어도 됩니다.)
eliminate_cafe = [
    # 'https://cafe.naver.com/thswndhks123',
    # 'https://cafe.naver.com/koarane',
    # 'https://cafe.naver.com/sjs986'
]

print(f"제외할 카페: {len(eliminate_cafe)}개")

제외할 카페: 0개


### 1.3 키워드 조합 생성

In [5]:
# 키워드 조합 생성
keywords = []

for main in main_keywords:
    for inv in invest_key:
        keywords.append(main + ' +%2B' + inv + ' ' + eliminate_ads_keyword)

print("생성된 키워드:")
for i, kw in enumerate(keywords, 1):
    print(f"  {i}. {kw}")

생성된 키워드:
  1. 귀농 +%2B건강식 
  2. 귀농 +%2B아침 


## 2. 헬퍼 함수 정의

In [6]:
def safe_close_extra_tabs(driver, keep_count=1):
    """
    안전하게 추가 탭들을 닫는 함수

    Args:
        driver: Selenium WebDriver
        keep_count: 유지할 탭 수 (기본값: 1)
    """
    try:
        max_attempts = 10  # 무한 루프 방지
        attempts = 0

        while len(driver.window_handles) > keep_count and attempts < max_attempts:
            driver.switch_to.window(driver.window_handles[-1])
            driver.close()
            time.sleep(0.3)
            attempts += 1

        driver.switch_to.window(driver.window_handles[0])
    except Exception as e:
        pass
        try:
            driver.switch_to.window(driver.window_handles[0])
        except:
            pass


def handle_alert(driver, timeout=2):
    """
    Alert 팝업 처리 함수

    Args:
        driver: Selenium WebDriver
        timeout: 대기 시간 (초)

    Returns:
        bool: Alert가 있었으면 True, 없었으면 False
    """
    try:
        WebDriverWait(driver, timeout).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        alert_text = alert.text
        logging.info(f"Alert 감지: {alert_text}")
        alert.accept()
        return True
    except (TimeoutException, NoAlertPresentException):
        return False


def switch_to_cafe_frame(driver, target_id=0, max_retries=3):
    """
    수집 대상에 따라 메인 프레임으로 전환하는 함수
    다음카페(target_id == 0)인 경우 down 프레임으로 전환하고, 그 외 대상은 기본 프레임 유지

    Args:
        driver: Selenium WebDriver
        target_id: 수집 대상 번호
        max_retries: 최대 재시도 횟수

    Returns:
        bool: 성공 여부
    """
    if target_id != 0:
        try:
            driver.switch_to.default_content()
            return True
        except Exception:
            return False

    for attempt in range(max_retries):
        try:
            # ★★★ 중요: 먼저 default content로 돌아가기 ★★★
            driver.switch_to.default_content()
            time.sleep(0.5)

            # 다음카페 iframe(down) 존재 확인
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "down"))
            )

            # 프레임 전환
            driver.switch_to.frame("down")
            return True
        except Exception as e:
            pass
            if attempt < max_retries - 1:
                driver.refresh()
                time.sleep(2)
    return False


def extract_post_data(driver, target_id=0):
    """
    게시물 데이터 추출 함수

    Args:
        driver: Selenium WebDriver
        target_id: 수집 대상 번호 (0: 다음카페, 1: 티스토리, 2: 브런치, 3: 웹문서, 4: 다음뉴스)

    Returns:
        dict or None: 추출된 데이터 또는 실패시 None
    """
    try:
        try:
            if "chrome-error://" in str(driver.current_url):
                return None
        except:
            pass
        soup = bs(driver.page_source, "lxml")

        # 0: 다음카페
        if target_id == 0:
            title_elem = soup.select_one("span.article_title, strong.tit_info, h3.tit_subject")
            if not title_elem:
                return None
            title = title_elem.get_text(strip=True)

            contents_elems = soup.select("div#user_contents, div.article_view")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems if i.get_text(strip=True)])
            if not contents:
                xmp_elem = soup.select_one("xmp#template_xmp")
                if xmp_elem and xmp_elem.string:
                    contents = bs(xmp_elem.string, "lxml").get_text(" ", strip=True)
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            date = ""
            for item in soup.select("div.cover_info span.txt_item, span.num_date, span.date"):
                txt = item.get_text(strip=True)
                if re.search(r"\d{2,4}\.\d{1,2}\.\d{1,2}", txt):
                    date = txt
                    break

            comments = [i.get_text(" ", strip=True) for i in soup.select("span.original_comment")]

            try:
                author_elem = soup.select_one("div.cover_info a.link_item, span.txt_writer")
                author_nickname = author_elem.get_text(strip=True) if author_elem else "닉네임 없음"
            except:
                author_nickname = "닉네임 없음"

            try:
                comment_nick_elems = soup.select("div.comment_view a.txt_name")
                if comment_nick_elems:
                    comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems])
                else:
                    comment_nicknames = "댓글 닉네임 없음"
            except:
                comment_nicknames = "댓글 닉네임 없음"

        # 1: 다음블로그_티스토리
        elif target_id == 1:
            og_title = soup.select_one("meta[property='og:title']")
            title_elem = soup.select_one("div.hgroup h1, h2.tit_post, .titleWrap h2, h1")
            title = og_title["content"].strip() if (og_title and og_title.get("content")) else (title_elem.get_text(strip=True) if title_elem else "")
            if not title:
                return None

            contents_elems = soup.select("div.contents_style, div.tt_article_useless_p_margin, div.article-view, div.entry-content, article")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time'], meta[property='og:regDate']")
            date_elem = soup.select_one("span.date, time")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("div.rp_general p, div.comment-content, p.txt_reply, div.tt-box-content .tt-desc")]

            author_meta = soup.select_one("meta[name='author'], meta[property='og:site_name']")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else "닉네임 없음"

            comment_nick_elems = soup.select("span.name, .tt-box-meta .tt-link-user, .rp_general .name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 2: 다음블로그_브런치
        elif target_id == 2:
            title_elem = soup.select_one("h1.cover_title")
            og_title = soup.select_one("meta[property='og:title']")
            title = title_elem.get_text(strip=True) if title_elem else (og_title["content"].strip() if (og_title and og_title.get("content")) else "")
            if not title:
                return None

            contents_elems = soup.select("div.wrap_body, div.wrap_body_frame")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time'], meta[property='og:regDate']")
            date_elem = soup.select_one("span.f_nb, span.date")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("div.comment_content p.desc_comment, .list_comment .desc_comment")]

            author_meta = soup.select_one("meta[name='author']")
            author_elem = soup.select_one("span.by_user a, strong.author_name")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else (author_elem.get_text(strip=True) if author_elem else "닉네임 없음")

            comment_nick_elems = soup.select(".list_comment .link_nick, .comment_head .name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 3: 다음웹문서
        elif target_id == 3:
            og_title = soup.select_one("meta[property='og:title']")
            title_elem = soup.select_one("h1, h2, title")
            title = og_title["content"].strip() if (og_title and og_title.get("content")) else (title_elem.get_text(strip=True) if title_elem else "")
            if not title:
                return None

            contents_elems = soup.select("article, main, div.write_div, div.rd_body, div.post_content, div.view_content, div#body")
            if not contents_elems:
                contents_elems = soup.select("body p")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            pub_meta = soup.select_one("meta[property='article:published_time']")
            date_elem = soup.select_one("time, span.date, span.gall_date")
            date = pub_meta["content"].strip() if (pub_meta and pub_meta.get("content")) else (date_elem.get_text(strip=True) if date_elem else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select(".ub-content .usertxt, .comment_content, .fdb_lst_ul .xe_content")]

            author_meta = soup.select_one("meta[name='author']")
            author_elem = soup.select_one(".gall_writer .nickname, .member_plate, .author")
            author_nickname = author_meta["content"].strip() if (author_meta and author_meta.get("content")) else (author_elem.get_text(strip=True) if author_elem else "닉네임 없음")

            comment_nick_elems = soup.select(".ub-writer .nickname, .comment_nick, .meta .author")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        # 4: 다음뉴스
        elif target_id == 4:
            title_elem = soup.select_one("h3.tit_view")
            og_title = soup.select_one("meta[property='og:title']")
            title = title_elem.get_text(strip=True) if title_elem else (og_title["content"].strip() if (og_title and og_title.get("content")) else "")
            if not title:
                return None

            contents_elems = soup.select("div.article_view section, div.article_view")
            contents = " ".join([i.get_text(" ", strip=True) for i in contents_elems[:1]])
            contents = contents.replace("\u200b", " ")
            contents = re.sub(r"\s+", " ", contents).strip()

            date_elem = soup.select_one("span.num_date")
            reg_meta = soup.select_one("meta[property='og:regDate']")
            date = date_elem.get_text(strip=True) if date_elem else (reg_meta["content"].strip() if (reg_meta and reg_meta.get("content")) else "")

            comments = [i.get_text(" ", strip=True) for i in soup.select("ul.list_comment p.desc_txt")]

            author_elem = soup.select_one("span.txt_info")
            author_nickname = author_elem.get_text(strip=True) if author_elem else "닉네임 없음"

            comment_nick_elems = soup.select("ul.list_comment a.link_nick, ul.list_comment span.txt_name")
            comment_nicknames = " | ".join([n.get_text(strip=True) for n in comment_nick_elems]) if comment_nick_elems else "댓글 닉네임 없음"

        else:
            return None

        error_keywords = ["사이트에 연결할 수 없음", "연결이 비공개로 설정되어 있지 않습니다", "ERR_"]
        if any(err in title for err in error_keywords):
            return None

        return {
            "title": title,
            "contents": contents,
            "date": date,
            "comments": comments,
            "author_nickname": author_nickname,
            "comment_nicknames": comment_nicknames
        }
    except Exception as e:
        pass
        return None


print("헬퍼 함수 정의")

헬퍼 함수 정의


## 3. 병렬배치 설정
- 설정값에 따라 속도를 배로 올릴 수 있습니다.
- 컴퓨터가 뜨거워질 수 있습니다.

In [7]:
# 맥미니 M4 16GB RAM 기준 최적 다중배치 브라우저 수 설정
MAX_WORKERS = 4

def get_chrome_options():
    options = Options()
    options.add_argument("--disable-backgrounding-occluded-windows")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # 1번 설정 섹션의 USE_HEADLESS 값 적용
    if globals().get("USE_HEADLESS", False):
        options.add_argument("--headless=new")
        
    return options

def create_driver():
    return webdriver.Chrome(options=get_chrome_options())

print(f"WebDriver 다중배치 설정 완료 (동시 실행 브라우저 수: {MAX_WORKERS}개, 백그라운드 실행: {globals().get('USE_HEADLESS', False)})")

WebDriver 다중배치 설정 완료 (동시 실행 브라우저 수: 4개, 백그라운드 실행: True)


## 4. 제외 URL 설정
- 이미 수집한 CSV 파일 속 URL을 추출하여 수집대상에서 제외합니다.
- 기존에 수집 완료된 csv파일은 overlap_data 폴더에 두시면 됩니다.

In [8]:
# overlap_data 폴더 내 CSV 파일 로드 → 제외할 URL 추출
import urllib.parse

def normalize_url(url):
    """URL에서 쿼리스트링 제거 후 기본 경로만 추출"""
    if pd.isna(url):
        return ""
    parsed = urllib.parse.urlparse(str(url))
    # scheme + netloc + path 만 남김 (? 이후 파라미터 제거)
    return f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

existing_urls = set()
overlap_csv_files = glob.glob(os.path.join(OVERLAP_PATH, "*.csv"))
total_loaded_rows = 0

for csv_file in overlap_csv_files:
    try:
        df_existing = pd.read_csv(csv_file, encoding="utf-8-sig")
        if "url" in df_existing.columns:
            total_loaded_rows += len(df_existing)
            existing_urls.update(df_existing["url"].dropna().apply(normalize_url))
            print(f"로드 완료: {os.path.basename(csv_file)} ({len(df_existing)}행)")
        else:
            print(f"[경고] url 컬럼 없음 (스킵): {os.path.basename(csv_file)}")
    except Exception as e:
        print(f"[오류] 파일 읽기 오류 ({os.path.basename(csv_file)}): {e}")

existing_urls.discard("")
print(f"overlap_data 내 CSV 파일 수: {len(overlap_csv_files)}개 (총 {total_loaded_rows}행)")
print(f"제외할 고유 URL 수: {len(existing_urls)}개")

overlap_data 내 CSV 파일 수: 0개 (총 0행)
제외할 고유 URL 수: 0개


In [9]:
# 데이터프레임 초기화
dataframe = pd.DataFrame(columns=['date', 'keyword', 'title', 'contents', 'comments', 'site', 'url', 'author_nickname', 'comment_nicknames'])
index = 0

print("데이터프레임 초기화 완료!")

데이터프레임 초기화 완료!


## 5. 크롤링 실행

- 한번에 검색 가능한 페이지 수에 한계가 있는 듯 합니다.
- 따라서 월별로 분할하여 데이터를 저장합니다.
- 최대 target은 월별 데이터를 다 합쳐서 키워드별로 3000개입니다. 따로 변경 하실 수 있습니다.
- 5회 연속으로 저장에 실패하면 다음 월 구간으로 넘어갑니다.(해당 월에는 정보가 없다는 뜻입니다.)

In [ ]:
TARGET_COUNT = 3000
MAX_CONSECUTIVE_ERRORS = 5

# 수집 기간: 20241010 ~ 20260630 (월별 구간 분할, 구간 내 관련도순 정렬)
YEAR_RANGES = [
    ("20260601", "20260630"),
    ("20260501", "20260531"),
    ("20260401", "20260430"),
    ("20260301", "20260331"),
    ("20260201", "20260228"),
    ("20260101", "20260131"),
    ("20251201", "20251231"),
    ("20251101", "20251130"),
    ("20251001", "20251031"),
    ("20250901", "20250930"),
    ("20250801", "20250831"),
    ("20250701", "20250731"),
    ("20250601", "20250630"),
    ("20250501", "20250531"),
    ("20250401", "20250430"),
    ("20250301", "20250331"),
    ("20250201", "20250228"),
    ("20250101", "20250131"),
    ("20241201", "20241231"),
    ("20241101", "20241130"),
    ("20241010", "20241031"),
]

def extract_search_links(page_source, w_param):
    """Daum 검색 결과 페이지에서 게시글 링크 목록 추출"""
    check_soup = bs(page_source, "lxml")
    if w_param == "fusion":
        links = [el.get("data-href") for el in check_soup.select("c-title[data-href]") if el.get("data-href")]
    else:
        links = [el.get("href") for el in check_soup.select(".c-list-basic .item-title a, .tit-g a") if el.get("href")]
    return links

def crawl_task(target_id, key, disp_handle=None):
    """단일 (수집대상, 키워드) 조합을 독립 브라우저에서 크롤링하는 작업 함수"""
    target_info = DAUM_TARGETS[target_id]
    target_name = target_info["name"]
    w_param = target_info["w"]
    col_param = target_info["col"]

    if disp_handle:
        disp_handle.update(f"[{target_name} | {key}] 현재 수집: 0개, 실패: 0개")

    try:
        driver = create_driver()
    except Exception as e:
        logging.error(f"[{target_name} | {key}] 브라우저 실행 실패: {e}")
        return None

    dataframe = pd.DataFrame(columns=["date", "keyword", "title", "contents", "comments", "site", "url", "author_nickname", "comment_nicknames"])
    index = 0
    current_collected = 0
    failed_count = 0


    try:
        for start_date_str, end_date_str in YEAR_RANGES:
            if current_collected >= TARGET_COUNT:
                break


            breaker = False
            page_num = 1
            consecutive_errors = 0
            max_page_limit = 11 if w_param == "fusion" else 100

            while not breaker:
                if len(dataframe) >= TARGET_COUNT or page_num > max_page_limit:
                    breaker = True
                    break

                if w_param == "fusion":
                    search_url = f"https://search.daum.net/search?w=fusion&col={col_param}&q={key}&sort=accuracy&period=u&sd={start_date_str}000000&ed={end_date_str}235959&p={page_num}"
                else:
                    search_url = f"https://search.daum.net/search?w={w_param}&q={key}&sort=accuracy&period=u&sd={start_date_str}000000&ed={end_date_str}235959&p={page_num}"

                driver.get(search_url)
                driver.implicitly_wait(15)
                time.sleep(2)

                href_list = extract_search_links(driver.page_source, w_param)

                if len(href_list) == 0:
                    breaker = True
                    break

                for href_value in href_list:
                    if len(dataframe) >= TARGET_COUNT:
                        breaker = True
                        break

                    try:
                        if any(element in href_value for element in eliminate_cafe):
                            continue

                        if normalize_url(href_value) in existing_urls:
                                continue

                        driver.execute_script(f"window.open('{href_value}', '_blank');")
                        time.sleep(random.uniform(0.25, 0.75))

                        try:
                            WebDriverWait(driver, 5).until(lambda d: len(d.window_handles) > 1)
                            driver.switch_to.window(driver.window_handles[-1])
                        except:
                            continue

                        time.sleep(random.uniform(0.25, 0.75))

                        if handle_alert(driver):
                            safe_close_extra_tabs(driver)
                            continue

                        if switch_to_cafe_frame(driver, target_id=target_id):
                            post_data = extract_post_data(driver, target_id=target_id)

                            if post_data and post_data["title"]:
                                driver.switch_to.default_content()
                                current_url = driver.current_url

                                if not dataframe["url"].apply(normalize_url).isin([normalize_url(current_url)]).any() and normalize_url(current_url) not in existing_urls:
                                    input_data = [
                                        post_data["date"],
                                        key,
                                        post_data["title"],
                                        post_data["contents"],
                                        post_data["comments"],
                                        target_name,
                                        current_url,
                                        post_data["author_nickname"],
                                        post_data["comment_nicknames"]
                                    ]
                                    dataframe.loc[index] = input_data
                                    index += 1
                                    current_collected += 1
                                    if disp_handle:
                                        disp_handle.update(f"[{target_name} | {key}] 현재 수집: {current_collected}개, 실패: {failed_count}개")
                                    consecutive_errors = 0
                            else:
                                consecutive_errors += 1
                                failed_count += 1
                                if disp_handle:
                                    disp_handle.update(f"[{target_name} | {key}] 현재 수집: {current_collected}개, 실패: {failed_count}개")
                        else:
                            consecutive_errors += 1
                            failed_count += 1
                            if disp_handle:
                                disp_handle.update(f"[{target_name} | {key}] 현재 수집: {current_collected}개, 실패: {failed_count}개")

                        safe_close_extra_tabs(driver)

                        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                            breaker = True
                            break

                    except Exception as e:
                        safe_close_extra_tabs(driver)
                        continue

                page_num += 1
                time.sleep(random.uniform(0.25, 0.75))

    finally:
        pass
        try:
            driver.quit()
        except:
            pass

    dataframe["norm_url"] = dataframe["url"].apply(normalize_url)
    dataframe = dataframe.drop_duplicates(subset="norm_url", keep="first").drop(columns=["norm_url"]).reset_index(drop=True)

    safe_keyword = key.replace("+%2B", "_").replace(" ", "").replace("/", "_")
    csv_filename = f"{SAVE_PATH}다음포탈_{target_name}_{safe_keyword}_{len(dataframe)}개.csv"

    dataframe.to_csv(csv_filename, encoding="utf-8-sig", index=False)
    if disp_handle:
        disp_handle.update(f"[{target_name} | {key}] 수집 완료: 총 {len(dataframe)}개 저장, 실패: {failed_count}개")
    return csv_filename

# ========== 다중배치 실행 큐 구성 ==========
tasks = []
print("========== 수집 실행 대상 최종 확인 ==========")
for target_id in SELECTED_TARGETS:
    if target_id in DAUM_TARGETS:
        print(f"수집 대상 [{target_id}]: {DAUM_TARGETS[target_id]['name']} (총 {len(keywords)}개 키워드)")
        for key in keywords:
            tasks.append((target_id, key))

print(f"총 {len(tasks)}개 작업을 최대 {MAX_WORKERS}개 다중배치로 실행합니다.\n")

completed_files = []
done_count = 0
total_tasks = len(tasks)

disp_handles = {}
for target_id, key in tasks:
    target_name = DAUM_TARGETS[target_id]["name"]
    disp_handles[(target_id, key)] = display(f"[{target_name} | {key}] 대기 중...", display_id=True)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(crawl_task, t_id, kw, disp_handles[(t_id, kw)]): (t_id, kw) for t_id, kw in tasks}
    for future in as_completed(futures):
        t_id, kw = futures[future]
        target_name = DAUM_TARGETS[t_id]["name"]
        done_count += 1
        try:
            res_file = future.result()
            if res_file:
                completed_files.append(res_file)
        except Exception as e:
            logging.error(f"[{done_count}/{total_tasks} 에러] 작업 실패 ({target_name} | {kw}): {e}")
print(f"\n전체 크롤링 완료: 총 {total_tasks}개 중 {len(completed_files)}개 파일 정상 저장됨")

========== 수집 실행 대상 최종 확인 ==========
수집 대상 [0]: 다음카페 (총 2개 키워드)
수집 대상 [1]: 다음블로그_티스토리 (총 2개 키워드)
수집 대상 [2]: 다음블로그_브런치 (총 2개 키워드)
총 6개 작업을 최대 4개 다중배치로 실행합니다.



'[다음카페 | 귀농 +%2B건강식 ] 수집 완료: 총 10개 저장'

'[다음카페 | 귀농 +%2B아침 ] 현재 수집: 28개'

'[다음블로그_티스토리 | 귀농 +%2B건강식 ] 현재 수집: 29개'

'[다음블로그_티스토리 | 귀농 +%2B아침 ] 현재 수집: 56개'

'[다음블로그_브런치 | 귀농 +%2B건강식 ] 수집 완료: 총 1개 저장'

'[다음블로그_브런치 | 귀농 +%2B아침 ] 현재 수집: 13개'

2026-10-27 15:53:40,168 - WARNING - Metadata file /Users/baenaongunmandu/.cache/selenium/se-metadata.json cannot be parsed: EOF while parsing a string at line 25 column 8. Using empty metadata


## 6. 전체 데이터 통합 파일 생성

- 수집 완료된 개별 CSV 파일들을 하나로 병합하여 중복을 제거한 `TOTAL` 파일을 생성합니다.
- 기존 개별 파일들은 그대로 유지되며, 통합 파일만 새로 추가됩니다.
- 크롤링이 모두 끝난 후 실행하거나, 나중에 단독으로 실행해도 됩니다.

In [ ]:
# # ========== 수집 완료된 개별 CSV 파일 통합 병합 ==========
# import glob
# import os
# import pandas as pd
#
# # TOTAL 파일을 제외한 개별 CSV 파일들 탐색
# csv_files = sorted([f for f in glob.glob(f"{SAVE_PATH}다음포탈_*.csv") if "_TOTAL_" not in f])
#
# if csv_files:
#     print(f"총 {len(csv_files)}개 개별 CSV 파일 통합 병합 시작...")
#     df_list = []
#     for f_path in csv_files:
#         try:
#             df_item = pd.read_csv(f_path, encoding="utf-8-sig")
#             df_list.append(df_item)
#             print(f"  - 로드 완료: {os.path.basename(f_path)} ({len(df_item)}행)")
#         except Exception as e:
#             print(f"  - 읽기 실패 ({os.path.basename(f_path)}): {e}")
#
#     if df_list:
#         total_df = pd.concat(df_list, ignore_index=True)
#
#         # URL 기준 중복 제거
#         if "normalize_url" in globals():
#             total_df["norm_url"] = total_df["url"].apply(normalize_url)
#             total_df = total_df.drop_duplicates(subset="norm_url", keep="first").drop(columns=["norm_url"]).reset_index(drop=True)
#         else:
#             total_df = total_df.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
#
#         total_csv_filename = f"{SAVE_PATH}다음포탈_TOTAL_{len(total_df)}개.csv"
#         total_df.to_csv(total_csv_filename, encoding="utf-8-sig", index=False)
#         print(f"
# 통합 파일 생성 완료: {total_csv_filename} (총 {len(total_df)}개 적재)
# ")
# else:
#     print("[경고] 통합할 개별 CSV 파일이 존재하지 않습니다.")
